# COLD-CUTS COSMED Viewer v1.0

This beginner-friendly notebook is a small interactive app for COSMED K5 Excel/CSV exports.

It detects the real data table inside messy exports, lets you choose variables, filters by time, defines simple event epochs, flags possible artifacts, and shows the results directly in the notebook app.

Important: possible speaking and movement artifacts are only rule-based review flags. They do not prove that speaking or movement happened.

## Step 1: Import the tools

This block loads Python packages and helper functions. The helper file keeps the notebook easier to read while still making the logic reusable.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cosmed_utils import (
    calculate_epoch_summary,
    calculate_summary_stats,
    convert_time_to_seconds,
    create_epochs_from_markers,
    detect_available_variables,
    load_cosmed_file,
    parse_manual_markers,
    plot_selected_variables,
    resample_data,
    run_qc_checks,
    smooth_data,
)

## Step 2: Create the upload and control panel

COSMED exports can come from a local file path or from an uploaded file. These controls let you load a file, select variables, set time filters, choose averaging and smoothing options, enter manual event markers, and decide which QC rules to run.

In [ ]:
file_path_box = widgets.Text(
    value=str(PROJECT_ROOT / "examples" / "synthetic_cosmed_export.xlsx"),
    description="File path:",
    layout=widgets.Layout(width="90%"),
)
upload_widget = widgets.FileUpload(accept=".xlsx,.csv,.txt,.tsv", multiple=False)
load_button = widgets.Button(description="Load file", button_style="primary")
run_button = widgets.Button(description="Run Analysis", button_style="success")

variable_select = widgets.SelectMultiple(
    options=[],
    value=(),
    description="Variables:",
    rows=10,
    layout=widgets.Layout(width="45%"),
)

start_time_box = widgets.Text(value="", description="Start:")
end_time_box = widgets.Text(value="", description="End:")

resample_dropdown = widgets.Dropdown(
    options=["raw/original", "1 second average", "5 second average", "10 second average", "30 second average", "60 second average"],
    value="raw/original",
    description="Refresh:",
)
smoothing_dropdown = widgets.Dropdown(
    options=["none", "rolling mean 5 samples", "rolling mean 10 samples", "rolling mean 30 samples"],
    value="none",
    description="Smooth:",
)

marker_text = widgets.Textarea(
    value="Baseline start, 00:30\nBaseline end, 02:30\nMovement start, 03:00\nMovement end, 04:00\nRecovery start, 04:30",
    description="Markers:",
    layout=widgets.Layout(width="90%", height="120px"),
)

flag_speaking_box = widgets.Checkbox(value=True, description="flag possible speaking")
flag_movement_box = widgets.Checkbox(value=True, description="flag possible movement")
flag_missing_box = widgets.Checkbox(value=True, description="flag missing values")
flag_jumps_box = widgets.Checkbox(value=True, description="flag sudden jumps")
flag_impossible_box = widgets.Checkbox(value=True, description="flag impossible values")

load_output = widgets.Output()
analysis_output = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<b>Load a COSMED file</b>"),
    file_path_box,
    upload_widget,
    load_button,
    load_output,
    widgets.HTML("<b>Analysis controls</b>"),
    widgets.HBox([variable_select, widgets.VBox([
        start_time_box,
        end_time_box,
        resample_dropdown,
        smoothing_dropdown,
        flag_speaking_box,
        flag_movement_box,
        flag_missing_box,
        flag_jumps_box,
        flag_impossible_box,
    ])]),
    marker_text,
    run_button,
    analysis_output,
]))

## Step 3: Load the COSMED file

COSMED exports often include metadata above the real table. This code searches the first 50 rows for the row that looks most like a COSMED header, then loads and cleans the table.

In [ ]:
STATE = {
    "df": None,
    "detection": None,
    "file_analyzed": None,
}

def save_uploaded_file() -> Path | None:
    if not upload_widget.value:
        return None
    uploaded = next(iter(upload_widget.value.values())) if isinstance(upload_widget.value, dict) else upload_widget.value[0]
    name = uploaded["metadata"]["name"] if "metadata" in uploaded else uploaded["name"]
    content = uploaded["content"]
    destination = PROJECT_ROOT / "examples" / name
    destination.write_bytes(content)
    return destination

def load_clicked(_):
    with load_output:
        clear_output()
        uploaded_path = save_uploaded_file()
        file_path = uploaded_path if uploaded_path else Path(file_path_box.value).expanduser()
        print(f"Loading: {file_path}")
        try:
            df, detection = load_cosmed_file(file_path)
        except Exception as error:
            print(f"Could not load file: {error}")
            return

        STATE["df"] = df
        STATE["detection"] = detection
        STATE["file_analyzed"] = str(file_path)

        available = detect_available_variables(df)
        variable_select.options = available
        variable_select.value = tuple(available[:4])

        print(f"Detected header row: {detection['header_row']}")
        print(f"Confidence score: {detection['confidence']:.2f}")
        print(f"Matched headers: {', '.join(detection['matched_headers'])}")
        print(f"Rows loaded: {len(df)}")
        print(f"Columns detected: {len(df.columns)}")

        if detection["confidence"] < 0.5:
            print("\nWarning: low confidence header detection. Please inspect the first 20 raw rows below.")
            display(detection["preview"].head(20))

        for col in ["Phase", "Marker"]:
            if col in df.columns:
                unique_values = sorted([str(value) for value in df[col].dropna().unique() if str(value).strip()])[:30]
                print(f"\nUnique {col} values:")
                print(unique_values)

        display(df.head())

load_button.on_click(load_clicked)

## Step 4: Run the analysis

This block filters by time, applies optional resampling and smoothing, parses manual event markers, runs QC checks, creates summaries and plots, then shows everything inside the notebook app.

In [ ]:
def parse_time_box(value):
    if not str(value).strip():
        return None
    parsed = convert_time_to_seconds(pd.Series([value])).iloc[0]
    return None if pd.isna(parsed) else float(parsed)

def run_clicked(_):
    with analysis_output:
        clear_output()
        if STATE["df"] is None:
            print("Please load a file first.")
            return

        df = STATE["df"].copy()
        selected = list(variable_select.value)
        start_time = parse_time_box(start_time_box.value)
        end_time = parse_time_box(end_time_box.value)

        if "t_seconds" in df.columns:
            if start_time is not None:
                df = df[df["t_seconds"] >= start_time]
            if end_time is not None:
                df = df[df["t_seconds"] <= end_time]

        df = resample_data(df, "t_seconds", resample_dropdown.value)
        df = smooth_data(df, selected, smoothing_dropdown.value)

        markers = parse_manual_markers(marker_text.value)
        epochs = create_epochs_from_markers(markers)

        summary = calculate_summary_stats(df, selected)
        epoch_summary = calculate_epoch_summary(df, selected, epochs)
        qc_report = run_qc_checks(
            df,
            selected,
            flag_speaking=flag_speaking_box.value,
            flag_movement=flag_movement_box.value,
            flag_missing=flag_missing_box.value,
            flag_jumps=flag_jumps_box.value,
            flag_impossible=flag_impossible_box.value,
        )

        fig = plot_selected_variables(df, selected, epochs, qc_report)

        print("Cleaned preview table")
        display(df.head(10))
        print("\nSummary statistics")
        display(summary)
        print("\nEpoch summary")
        display(epoch_summary if not epoch_summary.empty else pd.DataFrame({"message": ["No complete epochs were entered."]}))
        print("\nQC report preview")
        display(qc_report.head(20) if not qc_report.empty else pd.DataFrame({"message": ["No QC flags found."]}))
        display(fig)
        print("\nResults are shown here in the app. No CSV or Excel output files are needed for the normal workflow.")

run_button.on_click(run_clicked)

## Step 5: Optional one-click demo

If you are using the synthetic example file, click **Load file** and then **Run Analysis** above. The example intentionally includes a few abrupt changes so you can see the QC report and plot markers working.